# Capstone Project — Python Ops Automation Platform

**Level: Intermediate**

## Objective

Build a production-ready **`ops-bot`** CLI tool that integrates every phase of this course.

### What You Will Build

A Python CLI (`ops-bot`) with four commands:

| Command | What it does | Phases used |
|---------|-------------|-------------|
| `ops-bot inventory` | List AWS EC2 instances with Rich table output | 1, 2 |
| `ops-bot scan` | Run Bandit + pip-audit; exit 1 on HIGH findings | 1, 4 |
| `ops-bot report` | Run inventory + scan; publish Prometheus metrics; send Slack alert | 1–5 |
| `ops-bot serve` | Start Prometheus metrics HTTP server | 5 |

The platform is also packaged as a Docker container with a `docker-compose.yml`
that spins up `ops-bot + Prometheus + Grafana`.

### Rules
- Work through steps **in order** — each builds on the previous
- Refer to lessons when stuck, but try to implement first
- Complete the verification checklist before moving to stretch goals

**Estimated time:** 6–10 hours

## Prerequisites

Before starting, verify you have:

- [ ] Python 3.10+ and a virtual environment activated
- [ ] Docker Desktop running
- [ ] AWS credentials configured (`aws sts get-caller-identity` works), OR LocalStack for simulation
- [ ] All Phase 1–5 lessons completed or reviewed
- [ ] Slack incoming webhook URL (optional — alerts can be simulated)

**Reference lessons:**

| Step | Lesson |
|------|--------|
| Project structure | 1.1 |
| Config & .env | 1.3 |
| Click CLI | 1.4 |
| Structured logging | 1.5 |
| boto3 / EC2 | 2.1, 2.2 |
| Security scanning | 4.1, 4.2 |
| Prometheus metrics | 5.1 |
| Slack alerting | 5.5 |

## Step 1: Create the Project Structure

*Refer to: Lesson 1.1*

In [ ]:
from pathlib import Path
import subprocess, sys

root = Path("ops-bot")
dirs = [
    root / "src" / "ops_bot" / "cloud",
    root / "src" / "ops_bot" / "security",
    root / "src" / "ops_bot" / "monitoring",
    root / "src" / "ops_bot" / "alerting",
    root / "tests",
    root / "configs",
    root / "dashboards",
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    (d / "__init__.py").touch()

print("Project structure created:")
for f in sorted(root.rglob("*")):
    rel = f.relative_to(root)
    print(f"  {'  ' * (len(rel.parts)-1)}{'📁' if f.is_dir() else '📄'} {rel.name}")

## Step 2: Install Dependencies

In [ ]:
reqs = (
    "boto3>=1.28.0\n"
    "python-dotenv>=1.0.0\n"
    "pyyaml>=6.0.1\n"
    "click>=8.1.7\n"
    "rich>=13.7.0\n"
    "structlog>=24.1.0\n"
    "prometheus_client>=0.19.0\n"
    "requests>=2.31.0\n"
)
(root / "requirements.txt").write_text(reqs)

subprocess.run(
    [sys.executable, "-m", "pip", "install"] +
    [l for l in reqs.splitlines() if l and not l.startswith("#")] +
    ["--quiet"],
    capture_output=True
)
print("Dependencies installed.")

## Step 3: Configuration Module

*Refer to: Lesson 1.3*

In [ ]:
%%writefile ops-bot/src/ops_bot/config.py
"""Application configuration loaded from environment variables.

Reference: https://12factor.net/config
"""
import os
from dataclasses import dataclass, field
from dotenv import load_dotenv
from pathlib import Path

# Load .env from project root (won't override existing env vars)
load_dotenv(Path(__file__).parents[3] / ".env", override=False)

@dataclass
class AlertThresholds:
    stopped_instances: int = 1     # alert if >= N stopped instances
    security_findings_high: int = 0  # alert if > N HIGH findings

@dataclass
class AppConfig:
    environment: str = "development"
    log_level: str = "INFO"
    aws_region: str = "us-east-1"
    aws_account_id: str = ""
    slack_webhook_url: str = ""
    prometheus_port: int = 9090
    scrape_interval_seconds: int = 30
    thresholds: AlertThresholds = field(default_factory=AlertThresholds)

def load_config() -> AppConfig:
    """Load AppConfig from environment variables."""
    return AppConfig(
        environment=os.environ.get("ENVIRONMENT", "development"),
        log_level=os.environ.get("LOG_LEVEL", "INFO"),
        aws_region=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
        aws_account_id=os.environ.get("AWS_ACCOUNT_ID", ""),
        slack_webhook_url=os.environ.get("SLACK_WEBHOOK_URL", ""),
        prometheus_port=int(os.environ.get("PROMETHEUS_PORT", "9090")),
        scrape_interval_seconds=int(os.environ.get("SCRAPE_INTERVAL", "30")),
    )


## Step 4: Logging Module

*Refer to: Lesson 1.5*

In [ ]:
%%writefile ops-bot/src/ops_bot/logger.py
"""Structured logging for ops-bot.

Reference: https://www.structlog.org/en/stable/
"""
import os
import structlog

def configure_logging(level: str = "INFO") -> None:
    environment = os.environ.get("ENVIRONMENT", "development")
    procs = [
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.dev.set_exc_info,
    ]
    if environment in ("production", "staging", "ci"):
        procs.append(structlog.processors.JSONRenderer())
    else:
        procs.append(structlog.dev.ConsoleRenderer())
    structlog.configure(
        processors=procs,
        wrapper_class=structlog.stdlib.BoundLogger,
        context_class=dict,
        logger_factory=structlog.PrintLoggerFactory(),
    )

def get_logger(name: str = "ops-bot", **ctx) -> structlog.stdlib.BoundLogger:
    return structlog.get_logger(name).bind(**ctx)


## Step 5: AWS Inventory Module

*Refer to: Lessons 2.1 – 2.2*

In [ ]:
%%writefile ops-bot/src/ops_bot/cloud/aws.py
"""AWS cloud inventory using boto3.

Reference:
  Quickstart:  https://docs.aws.amazon.com/boto3/latest/guide/quickstart.html
  Paginators:  https://docs.aws.amazon.com/boto3/latest/guide/paginators.html
  EC2:         https://docs.aws.amazon.com/boto3/latest/guide/ec2-example-managing-instances.html
"""
import os
import boto3
from botocore.exceptions import ClientError, NoCredentialsError

class AWSInventory:
    """EC2 and S3 inventory using boto3 with full pagination."""

    def __init__(self, region: str | None = None, session: boto3.Session | None = None):
        self.region = region or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
        self._session = session or boto3.Session(region_name=self.region)
        self._ec2 = self._session.client("ec2")
        self._s3  = self._session.client("s3")

    def list_ec2_instances(self, filters: list | None = None) -> list[dict]:
        """List ALL EC2 instances using pagination.

        Uses get_paginator() so every page is fetched automatically.
        Reference: https://docs.aws.amazon.com/boto3/latest/guide/paginators.html
        """
        paginator = self._ec2.get_paginator("describe_instances")
        kwargs = {"Filters": filters} if filters else {}
        instances = []
        for page in paginator.paginate(**kwargs):
            for r in page["Reservations"]:
                for inst in r["Instances"]:
                    tags = {t["Key"]: t["Value"] for t in inst.get("Tags", [])}
                    instances.append({
                        "instance_id": inst["InstanceId"],
                        "name":        tags.get("Name", "(unnamed)"),
                        "state":       inst["State"]["Name"],
                        "type":        inst.get("InstanceType", ""),
                        "private_ip":  inst.get("PrivateIpAddress", ""),
                        "tags":        tags,
                        "region":      self.region,
                    })
        return instances

    def list_s3_buckets(self) -> list[dict]:
        """List all S3 buckets.""",
        r = self._s3.list_buckets()
        return [{"name": b["Name"],
                 "created": b["CreationDate"].isoformat()}
                for b in r.get("Buckets", [])]

    def get_stopped_instances(self) -> list[dict]:
        return [i for i in self.list_ec2_instances()
                if i["state"] not in ("running", "pending")]


## Step 6: Security Scanner

*Refer to: Lessons 4.1 – 4.2*

In [ ]:
%%writefile ops-bot/src/ops_bot/security/scanner.py
"""SAST and dependency vulnerability scanning.

Reference:
  Bandit:    https://bandit.readthedocs.io/en/latest/
  pip-audit: https://pypi.org/project/pip-audit/
"""
import subprocess, json, re
from pathlib import Path

class OpsSecurityScanner:
    """Runs Bandit SAST + pip-audit CVE scanning."""

    def run_bandit(self, target: str) -> dict:
        """Run Bandit SAST scanner and return parsed results.""",
        r = subprocess.run(
            ["bandit", "-r", "-ll", "-f", "json", target],
            capture_output=True, text=True,
        )
        try:
            raw = json.loads(r.stdout)
        except json.JSONDecodeError:
            return {"error": r.stderr, "total": 0, "high": 0, "issues": []}
        issues = raw.get("results", [])
        return {
            "total": len(issues),
            "high":  sum(1 for i in issues if i["issue_severity"] == "HIGH"),
            "medium":sum(1 for i in issues if i["issue_severity"] == "MEDIUM"),
            "issues": [{"id": i["test_id"], "severity": i["issue_severity"],
                        "text": i["issue_text"], "file": i["filename"],
                        "line": i["line_number"]} for i in issues],
        }

    def run_pip_audit(self, requirements_file: str | None = None) -> dict:
        """Run pip-audit CVE scanner.""",
        cmd = ["pip-audit", "--format", "json", "--progress-spinner", "off"]
        if requirements_file:
            cmd.extend(["-r", requirements_file])
        r = subprocess.run(cmd, capture_output=True, text=True)
        try:
            raw = json.loads(r.stdout)
        except json.JSONDecodeError:
            return {"error": r.stderr, "cve_count": 0, "cves": []}
        cves = [{"pkg": d["name"], "ver": d["version"],
                 "id": v["id"], "fix": v.get("fix_versions", [])}
                for d in raw.get("dependencies", [])
                for v in d.get("vulns", [])]
        return {"cve_count": len(cves), "cves": cves}

    def has_blocking_issues(self, bandit: dict, audit: dict,
                            environment: str = "development") -> bool:
        """Return True if findings should block a deployment.""",
        if environment == "development":
            return bool(bandit.get("high", 0) > 5 or audit.get("cve_count", 0) > 10)
        # staging / production: stricter
        return bool(bandit.get("high", 0) > 0 or audit.get("cve_count", 0) > 0)


## Step 7: Prometheus Metrics Module

*Refer to: Lesson 5.1*

In [ ]:
%%writefile ops-bot/src/ops_bot/monitoring/metrics.py
"""Prometheus custom metrics for ops-bot.

Reference: https://github.com/prometheus/client_python
"""
import time
from prometheus_client import (
    Counter, Gauge, Histogram,
    CollectorRegistry, start_http_server,
)

class OpsMetrics:
    """Defines and exposes Prometheus metrics for the ops platform.""",

    def __init__(self, port: int = 9090):
        self.port = port
        self.registry = CollectorRegistry()

        self.instances = Gauge(
            "ops_instances_total", "Cloud instances by provider/region/state.",
            ["provider", "region", "state"], registry=self.registry,
        )
        self.stopped = Gauge(
            "ops_stopped_instances", "Stopped instance count.",
            ["provider", "region"], registry=self.registry,
        )
        self.scans = Counter(
            "ops_scans_total", "Total inventory scans performed.",
            ["provider", "status"], registry=self.registry,
        )
        self.security = Gauge(
            "ops_security_findings", "Security findings by tool/severity.",
            ["tool", "severity"], registry=self.registry,
        )
        self.compliance = Gauge(
            "ops_compliance_score", "Compliance score (0-100).",
            ["account", "framework"], registry=self.registry,
        )

    def update_instance_metrics(self, instances: list[dict], provider: str = "aws") -> None:
        running = sum(1 for i in instances if i.get("state") == "running")
        stopped = sum(1 for i in instances if i.get("state") != "running")
        region  = instances[0]["region"] if instances else "unknown"
        self.instances.labels(provider=provider, region=region, state="running").set(running)
        self.instances.labels(provider=provider, region=region, state="stopped").set(stopped)
        self.stopped.labels(provider=provider, region=region).set(stopped)
        self.scans.labels(provider=provider, status="success").inc()

    def update_security_metrics(self, bandit: dict, audit: dict) -> None:
        self.security.labels(tool="bandit",    severity="HIGH").set(bandit.get("high", 0))
        self.security.labels(tool="pip-audit", severity="CVE").set(audit.get("cve_count", 0))

    def start_server(self) -> None:
        start_http_server(self.port, registry=self.registry)
        print(f"✓  Metrics server: http://0.0.0.0:{self.port}/metrics")


## Step 8: Alerting Module

*Refer to: Lesson 5.5*

In [ ]:
%%writefile ops-bot/src/ops_bot/alerting/alerts.py
"""Slack webhook alerting with deduplication.

Reference: https://api.slack.com/messaging/webhooks
"""
import os, time, json, hashlib, requests

class OpsAlerter:
    def __init__(self, slack_webhook: str | None = None, dedup_seconds: int = 300):
        self.webhook = slack_webhook or os.environ.get("SLACK_WEBHOOK_URL", "")
        self.dedup_s = dedup_seconds
        self._fired: dict[str, float] = {}

    def _is_duplicate(self, key: str) -> bool:
        return time.time() - self._fired.get(key, 0) < self.dedup_s

    def send_slack(self, title: str, message: str, severity: str = "info",
                   fields: dict | None = None, alert_key: str | None = None) -> bool:
        key = alert_key or hashlib.md5(f"{title}{severity}".encode()).hexdigest()[:12]
        if self._is_duplicate(key):
            return False  # suppressed
        self._fired[key] = time.time()

        colors = {"info": "#2196F3", "warning": "#FF9800",
                  "error": "#F44336", "critical": "#9C27B0"}
        icons  = {"info": "ℹ️",  "warning": "⚠️", "error": "🔴", "critical": "🚨"}
        blocks = [
            {"type": "header", "text": {"type": "plain_text",
             "text": f"{icons.get(severity,'📣')} {title}"}},
            {"type": "section", "text": {"type": "mrkdwn", "text": message}},
        ]
        if fields:
            blocks.append({"type": "section",
                "fields": [{"type": "mrkdwn", "text": f"*{k}:* {v}"}
                            for k, v in fields.items()]})
        payload = {"text": f"[{severity.upper()}] {title}",
                   "attachments": [{"color": colors.get(severity, "#808080"),
                                    "blocks": blocks}]}
        if not self.webhook:
            print(f"[ALERT-SIM] {severity.upper()}: {title} | {message}")
            return True
        try:
            r = requests.post(self.webhook, json=payload, timeout=10)
            return r.status_code == 200
        except Exception:
            return False

    def check_and_alert(self, instances: list[dict], threshold: int = 1,
                        scan_result: dict | None = None) -> None:
        stopped = [i for i in instances if i.get("state") not in ("running", "pending")]
        if len(stopped) >= threshold:
            self.send_slack(
                title=f"{len(stopped)} EC2 instance(s) are stopped",
                message=f"Stopped instances detected in region {instances[0].get('region','?')}.",
                severity="warning" if len(stopped) < 5 else "error",
                fields={i["instance_id"]: i["name"] for i in stopped[:5]},
                alert_key=f"stopped-instances-{len(stopped)}",
            )
        if scan_result and scan_result.get("high", 0) > 0:
            self.send_slack(
                title=f"{scan_result['high']} HIGH severity security finding(s)",
                message="Bandit detected HIGH severity issues. Review before deploying.",
                severity="error",
                alert_key="bandit-high-findings",
            )


## Step 9: Build the CLI

*Refer to: Lesson 1.4*

In [ ]:
%%writefile ops-bot/src/ops_bot/cli.py
"""ops-bot CLI — Ops Automation Platform.

Usage:
    python -m ops_bot.cli inventory
    python -m ops_bot.cli scan --target src/
    python -m ops_bot.cli report --output report.json
    python -m ops_bot.cli serve --port 9090
"""
import click, json, sys, os
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

@click.group()
@click.option("--verbose", "-v", is_flag=True)
@click.option("--env", "-e", default=None,
              type=click.Choice(["development", "staging", "production"]))
@click.pass_context
def cli(ctx, verbose, env):
    """Ops Automation Platform CLI."""
    ctx.ensure_object(dict)
    ctx.obj["verbose"] = verbose
    ctx.obj["env"] = env or os.environ.get("ENVIRONMENT", "development")

    # Initialise logging
    from ops_bot.logger import configure_logging
    configure_logging()

@cli.command()
@click.option("--region", "-r", default=None)
@click.option("--output", "-o", default="table",
              type=click.Choice(["table", "json"]))
@click.pass_context
def inventory(ctx, region, output):
    """List AWS EC2 instance inventory."""
    from ops_bot.cloud.aws import AWSInventory
    from ops_bot.logger import get_logger
    log = get_logger("inventory")

    log.info("inventory_started", region=region or "default")
    try:
        inv = AWSInventory(region=region)
        instances = inv.list_ec2_instances()
    except Exception as e:
        console.print(f"[red]✗ Failed to fetch inventory: {e}[/red]")
        console.print("[dim]  Set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY, or use aws configure.[/dim]")
        instances = []  # graceful degradation

    if output == "json":
        click.echo(json.dumps(instances, indent=2, default=str))
        return

    table = Table(title=f"EC2 Inventory ({region or 'all regions'})")
    table.add_column("Instance ID", style="dim")
    table.add_column("Name")
    table.add_column("State")
    table.add_column("Type")
    table.add_column("Region")
    for i in instances:
        state_fmt = f"[green]{i['state']}[/green]" if i["state"] == "running" else f"[yellow]{i['state']}[/yellow]"
        table.add_row(i["instance_id"], i["name"], state_fmt, i["type"], i.get("region", ""))
    console.print(table)

    stopped = sum(1 for i in instances if i["state"] != "running")
    console.print(Panel(
        f"[bold]{len(instances)}[/bold] instances | "
        f"[green]{len(instances)-stopped} running[/green] | "
        f"[yellow]{stopped} stopped[/yellow]",
        title="Summary", border_style="cyan"
    ))
    log.info("inventory_completed", total=len(instances), stopped=stopped)

@cli.command()
@click.option("--target", "-t", default=".", help="Directory to scan")
@click.option("--dockerfile", default=None)
@click.pass_context
def scan(ctx, target, dockerfile):
    """Run DevSecOps security scans (Bandit + pip-audit)."""
    from ops_bot.security.scanner import OpsSecurityScanner
    from ops_bot.logger import get_logger
    log = get_logger("scan")
    env = ctx.obj.get("env", "development")
    scanner = OpsSecurityScanner()

    console.print(f"[bold]Running security scan[/bold] on [cyan]{target}[/cyan]")
    bandit = scanner.run_bandit(target)
    audit  = scanner.run_pip_audit()
    log.info("scan_complete", bandit_issues=bandit["total"], cves=audit["cve_count"])

    if bandit["total"] > 0:
        console.print(f"  [red]Bandit: {bandit['total']} issues ({bandit['high']} HIGH)[/red]")
    else:
        console.print("  [green]✓ Bandit: no issues[/green]")

    if audit["cve_count"] > 0:
        console.print(f"  [red]pip-audit: {audit['cve_count']} CVE(s)[/red]")
    else:
        console.print("  [green]✓ pip-audit: no CVEs[/green]")

    if scanner.has_blocking_issues(bandit, audit, env):
        console.print("[bold red]✗ Security gate FAILED[/bold red]")
        sys.exit(1)
    console.print("[bold green]✓ Security gate PASSED[/bold green]")

@cli.command()
@click.option("--format", "fmt", default="json", type=click.Choice(["json", "markdown"]))
@click.option("--output", default="report.json")
@click.pass_context
def report(ctx, fmt, output):
    """Run full ops report: inventory + scan + metrics + alerts."""
    import json
    from pathlib import Path
    from ops_bot.cloud.aws import AWSInventory
    from ops_bot.security.scanner import OpsSecurityScanner
    from ops_bot.monitoring.metrics import OpsMetrics
    from ops_bot.alerting.alerts import OpsAlerter
    from ops_bot.config import load_config
    from ops_bot.logger import get_logger

    cfg     = load_config()
    log     = get_logger("report", env=cfg.environment)
    metrics = OpsMetrics(port=cfg.prometheus_port)
    alerter = OpsAlerter(slack_webhook=cfg.slack_webhook_url)

    log.info("report_started")
    console.print("[bold]Ops Automation Platform — Full Report[/bold]")

    # Inventory
    console.print("\n[cyan]1. Cloud Inventory[/cyan]")
    try:
        inv = AWSInventory(region=cfg.aws_region)
        instances = inv.list_ec2_instances()
        metrics.update_instance_metrics(instances)
        console.print(f"   {len(instances)} instances found")
    except Exception as e:
        log.error("inventory_failed", error=str(e))
        instances = []

    # Security scan
    console.print("\n[cyan]2. Security Scan[/cyan]")
    scanner = OpsSecurityScanner()
    bandit  = scanner.run_bandit(".")
    audit   = scanner.run_pip_audit()
    metrics.update_security_metrics(bandit, audit)
    console.print(f"   Bandit: {bandit['high']} HIGH  | pip-audit: {audit['cve_count']} CVEs")

    # Alerts
    console.print("\n[cyan]3. Alerts[/cyan]")
    alerter.check_and_alert(instances, threshold=cfg.thresholds.stopped_instances,
                            scan_result=bandit)

    # Save report
    report_data = {
        "environment": cfg.environment,
        "instances": instances,
        "security": {"bandit": bandit, "pip_audit": audit},
    }
    Path(output).write_text(json.dumps(report_data, indent=2, default=str))
    console.print(f"\n[green]✓ Report saved: {output}[/green]")
    log.info("report_completed", output=output)

@cli.command()
@click.option("--port", default=9090, type=int)
@click.pass_context
def serve(ctx, port):
    """Start Prometheus metrics exporter."""
    import time
    from ops_bot.monitoring.metrics import OpsMetrics
    from ops_bot.cloud.aws import AWSInventory
    from ops_bot.security.scanner import OpsSecurityScanner
    from ops_bot.config import load_config

    cfg     = load_config()
    metrics = OpsMetrics(port=port)
    metrics.start_server()
    console.print(f"[bold]Collecting metrics every {cfg.scrape_interval_seconds}s...[/bold]")

    while True:
        try:
            inv = AWSInventory(region=cfg.aws_region)
            metrics.update_instance_metrics(inv.list_ec2_instances())
        except Exception:
            pass
        time.sleep(cfg.scrape_interval_seconds)

if __name__ == "__main__":
    cli()


## Step 10: Tests

In [ ]:
%%writefile ops-bot/tests/test_ops_bot.py
"""Unit tests for ops-bot."""
import pytest, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent.parent / "src"))

def test_config_loads_defaults():
    from ops_bot.config import load_config
    cfg = load_config()
    assert cfg.environment in ("development", "staging", "production")
    assert cfg.prometheus_port > 0
    assert cfg.scrape_interval_seconds > 0
    print(f"✓ config loaded: {cfg.environment}")

def test_alerter_deduplication():
    from ops_bot.alerting.alerts import OpsAlerter
    alerter = OpsAlerter(slack_webhook=None, dedup_seconds=60)
    r1 = alerter.send_slack("Test", "message", alert_key="test-key-001")
    r2 = alerter.send_slack("Test", "message", alert_key="test-key-001")
    assert r1 is True
    assert r2 is False   # deduplicated
    print("✓ deduplication works")

def test_security_scanner_blocking():
    from ops_bot.security.scanner import OpsSecurityScanner
    scanner = OpsSecurityScanner()
    # 0 issues + 0 CVEs should never block
    ok = not scanner.has_blocking_issues(
        {"high": 0, "total": 0},
        {"cve_count": 0},
        environment="production",
    )
    assert ok
    # 1 HIGH in production should block
    blocked = scanner.has_blocking_issues(
        {"high": 1, "total": 1},
        {"cve_count": 0},
        environment="production",
    )
    assert blocked
    print("✓ security gate thresholds correct")

if __name__ == "__main__":
    test_config_loads_defaults()
    test_alerter_deduplication()
    test_security_scanner_blocking()
    print("\nAll basic tests passed.")


In [ ]:
# Run the tests
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "pytest", "ops-bot/tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
    cwd="."
)
print(r.stdout)
if r.returncode == 0:
    print("\n✅  All tests passed!")
else:
    print("\n⚠   Some tests may be failing — implement the TODO stubs above.")

## Step 11: Dockerfile

In [ ]:
%%writefile ops-bot/Dockerfile
# ops-bot — Security-hardened multi-stage Dockerfile
# Reference: https://docs.docker.com/develop/security-best-practices/

# ── Stage 1: build dependencies ─────────────────────────────────────────
FROM python:3.11-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --user -r requirements.txt

# ── Stage 2: runtime ────────────────────────────────────────────────────
FROM python:3.11-slim

# Create a non-root user (CIS Docker Benchmark — no root in containers)
RUN groupadd -r opsbot && useradd -r -g opsbot -s /sbin/nologin opsbot

WORKDIR /app
COPY --from=builder /root/.local /home/opsbot/.local
COPY src/ ./src/

RUN chown -R opsbot:opsbot /app
USER opsbot

EXPOSE 9090

# Health check: poll the /metrics endpoint every 30s
HEALTHCHECK --interval=30s --timeout=10s --start-period=10s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:9090/metrics')" || exit 1

ENV PYTHONPATH=/app/src
CMD ["python", "-m", "ops_bot.cli", "serve"]


## Step 12: docker-compose.yml

In [ ]:
%%writefile ops-bot/docker-compose.yml
version: '3.8'

services:
  ops-bot:
    build: .
    ports:
      - "9090:9090"
    environment:
      - ENVIRONMENT=development
      - LOG_LEVEL=INFO
      - AWS_DEFAULT_REGION=${AWS_DEFAULT_REGION:-us-east-1}
      - AWS_ACCESS_KEY_ID=${AWS_ACCESS_KEY_ID:-}
      - AWS_SECRET_ACCESS_KEY=${AWS_SECRET_ACCESS_KEY:-}
      - SLACK_WEBHOOK_URL=${SLACK_WEBHOOK_URL:-}
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9091:9090"
    volumes:
      - ./configs/prometheus.yml:/etc/prometheus/prometheus.yml:ro
    restart: unless-stopped

  grafana:
    image: grafana/grafana:latest
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=ops-admin
    volumes:
      - grafana-data:/var/lib/grafana
    restart: unless-stopped
    depends_on:
      - prometheus

volumes:
  grafana-data:


In [ ]:
(Path("ops-bot/configs")).mkdir(exist_ok=True)
(Path("ops-bot/configs/prometheus.yml")).write_text(
    "global:\n  scrape_interval: 30s\n\n"
    "scrape_configs:\n"
    "  - job_name: ops-bot\n"
    "    static_configs:\n"
    "      - targets: [ops-bot:9090]\n"
)
print("✓ prometheus.yml written")

## Verification Checklist

Mark each item complete before tackling stretch goals:

- [ ] **Step 1:** Project structure with all dirs and `__init__.py` files
- [ ] **Step 2:** `requirements.txt` written and all deps installable
- [ ] **Step 3:** `config.py` — `AppConfig` loads from env vars
- [ ] **Step 4:** `logger.py` — `configure_logging()` and `get_logger()` work
- [ ] **Step 5:** `cloud/aws.py` — `AWSInventory` uses `get_paginator()`
- [ ] **Step 6:** `security/scanner.py` — bandit + pip-audit + blocking logic
- [ ] **Step 7:** `monitoring/metrics.py` — Prometheus metrics defined + HTTP server starts
- [ ] **Step 8:** `alerting/alerts.py` — Slack alerts with deduplication
- [ ] **Step 9:** `cli.py` — all 4 sub-commands work
- [ ] **Step 10:** 3+ unit tests pass with `pytest`
- [ ] **Step 11:** Dockerfile builds without errors
- [ ] **Step 12:** `docker-compose up` starts ops-bot + Prometheus + Grafana

## Stretch Goals

### 1. Multi-Cloud Inventory (Phase 2)
Extend `inventory` to include Azure VMs and GCP instances using the adapter pattern from Lesson 2.5.

### 2. GitOps Integration (Phase 3)
Add `ops-bot deploy` that updates a Helm values file and opens a GitHub PR.

### 3. Full DevSecOps CI/CD Pipeline (Phase 4)
Add a GitHub Actions workflow that runs `ops-bot scan` on every PR and fails
on HIGH findings.

### 4. Distributed Tracing (Phase 5)
Instrument `ops-bot` commands with OpenTelemetry spans (Lesson 5.2) and add
Jaeger to `docker-compose.yml`.

### 5. Grafana Dashboard Auto-Provisioning
Use the Grafana API (Lesson 5.6) to deploy the ops dashboard on startup.

---

**Congratulations on completing the Python for CloudOps + DevOps + DevSecOps + Monitoring study plan!**